In [ ]:
# %pip install --upgrade transformers --break-system-packages

In [ ]:
# %pip install auto-round --break-system-packages

In [ ]:
# %pip install git+https://github.com/sustcsonglin/flash-linear-attention.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install git+https://github.com/Dao-AILab/causal-conv1d.git --no-build-isolation --break-system-packages

In [ ]:
# %pip install compressed-tensors --break-system-packages

In [ ]:
import os

import torch
from auto_round import AutoRound, SignRoundConfig
from huggingface_hub import HfApi, create_repo, get_token, notebook_login
from safetensors import safe_open
from transformers import AutoModelForImageTextToText, AutoProcessor


In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


In [ ]:
notebook_login()

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-9B"
HF_USER = "your_hf_username"  # Replace with your Hugging Face username
OUTPUT_BASE_DIR = "./AutoRound"
LOCAL_PATH = "./local_model"

In [ ]:
!hf download $MODEL_ID --local-dir $LOCAL_PATH

In [ ]:
model = AutoModelForImageTextToText.from_pretrained(
    LOCAL_PATH, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(LOCAL_PATH)

tokenizer = processor.tokenizer


In [ ]:

# Configure optimization parameters for the AutoRound / SignRound algorithm
alg_config = SignRoundConfig(
    iters=1000,      # Optimization iterations per block; balances calibration time and accuracy
    group_size=16,   # Number of weights sharing one scale. Use 16 when targeting NVFP4.
    sym=True,        # Symmetric quantization; enables faster compute and broader engine compatibility
)

# Initialize the AutoRound quantization pipeline
ar = AutoRound(
    model=model,                                # Target pretrained model to be quantized
    tokenizer=tokenizer,                        # Tokenizer used to process text calibration samples
    processor=processor,                        # Processor for multimodal models (vision/audio), if present
    scheme="NVFP4",                             # NVIDIA FP4 quantization format for reduced memory usage and accelerated inference on supported hardware
    alg_configs=alg_config,                     # Applies the algorithm-specific optimization settings defined above
    nsamples=512,                               # Total calibration dataset samples to maintain quality and avoid overfitting
    batch_size=4,                               # Calibration batch size; balances processing speed and GPU memory usage
    gradient_accumulate_steps=2,                # Set to 8 / batch_size to maintain an effective global batch size of 8; smaller values risk accuracy drop    
    seqlen=2048,                                # Calibration context length; captures long-range dependency activations
    low_gpu_mem_usage=False,                    # Keeps modules on GPU to maximize speed if sufficient VRAM is available
    quant_nontext_module=False,                 # Leaves non-text modules (e.g., vision encoders) unquantized to protect precision
    enable_torch_compile=False,                 # Disables torch.compile to avoid the NVFP4/static warning
    ignore_layers="linear_attn,mtp,visual",     # Excludes linear-attention, MTP, and visual modules from quantization
    layer_config={
        "mtp": {"data_type": "bfloat16"},       # Keeps the MTP module in BF16 precision
        "mtp.fc": {"data_type": "bfloat16"},    # Keeps the MTP fully connected layer in BF16 precision
    },                                          # Preserves BF16 precision for the specified MTP layers
)


In [ ]:
# Quantize once and export to disk using the recommended schema for NVFP4
ar.quantize_and_save(
    OUTPUT_BASE_DIR,
    # Use 'llm_compressor' for the compressed-tensors schema, recommended for NVFP4 inference with vLLM / Neural Magic
    format="llm_compressor",
    # Overwrites weights in-place to avoid extra VRAM allocation
    inplace=True,
)


In [ ]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()

In [ ]:
if hf_token:
    push_to_hub(
        os.path.join(OUTPUT_BASE_DIR, "local_model-nvfp-w4g16"),
        f"{base_name}-W4A16-AutoRound-NVFP4",
        hf_token,
    )
else:
    print("No Hugging Face token found. Skipping upload to hub.")